# 0909 8일차

## 0. 파이썬 문법: 클래스 인스턴스화와 메서드 체이닝

`OneHotEncoder`를 쓰다 나온 에러

```python
y = OneHotEncoder.fit_transform.toarray(y)
```
```
AttributeError: 'function' object has no attribute 'toarray'
```

→ 두 가지가 어긋나 있음

### ① 클래스 - 괄호를 붙여야 객체가 됨

```python
OneHotEncoder          # 클래스 그 자체 (설계도)
OneHotEncoder()        # 객체 (실제 물건)
```

→ 괄호 없이 쓰면 `self`가 없어 메서드를 호출할 수 없음

| 이름 | 정체 | 호출 |
|---|---|---|
| `to_categorical` | 소문자 → **함수** | `to_categorical(y)` |
| `OneHotEncoder` | 대문자 → **클래스** | `OneHotEncoder().fit_transform(y)` |

→ 2일차 §0의 이름 규칙(PEP8)이 그대로 힌트가 됨

### ② 메서드 체이닝 - 결과에 이어서 호출

```python
ohe.fit_transform(y).toarray()
    └─ ①실행 ─┘  └─ ②그 결과에 호출 ─┘
```

→ 왼쪽부터 차례로 실행되고 **앞의 반환값**에 다음 메서드가 붙음

```python
ohe.fit_transform.toarray(y)    # X - 메서드에 메서드를 붙일 수 없음

result = ohe.fit_transform(y)   # 나눠 쓰면 이 뜻
result.toarray()
```

→ `pd.get_dummies(y).values`, `df.dropna().shape`도 같은 구조

## 1. 다중분류 - 라벨이 3개 이상

7일차의 이진분류(라벨 2개)에서 라벨이 늘어난 경우

| | 라벨 | 출력층 | loss |
|---|---|---|---|
| **이진분류** | 2개 | `Dense(1, activation='sigmoid')` | `binary_crossentropy` |
| **다중분류** | 3개 이상 | `Dense(라벨수, activation='softmax')` | `categorical_crossentropy` |

→ 출력 노드가 **라벨 개수만큼** 늘어남. iris는 3개, digits는 10개

## 2. 원핫 인코딩 - 필요한 이유

y를 그대로 두고 `Dense(3, activation='softmax')`를 돌리면

```
ValueError: Arguments `target` and `output` must have the same rank (ndim).
            Received: target.shape=(8,), output.shape=(8, 3)
```

→ 출력은 `(8, 3)`인데 정답은 `(8,)` 정수라 **rank(차원 수)가 안 맞음**

→ `categorical_crossentropy`는 **정답도 원핫이길 기대**함

### 라벨 간 거리 - 가치를 동일하게

라벨을 `0, 1, 2` 숫자로 두면 **없던 거리 관계가 생김**

```
|0 - 1| = 1,  |1 - 2| = 1,  |0 - 2| = 2
```

→ 붓꽃 품종에는 순서가 없는데, 모델은 "2는 1보다 크다", "1은 0과 2의 중간"으로 학습할 수 있음

**원핫으로 바꾸면 거리가 전부 같아짐**

```
라벨 0 <-> 1 : 1.4142
라벨 0 <-> 2 : 1.4142
라벨 1 <-> 2 : 1.4142
```

→ **데이터를 위치값으로** 보는 것. `2`라는 크기가 아니라 "세 자리 중 3번째"라는 위치

```
[0, 0, 1, 1, 2]     (5,)
->
[[1, 0, 0],
 [1, 0, 0],
 [0, 1, 0],
 [0, 1, 0],
 [0, 0, 1]]         (5, 3)
```

### cf) 원핫과 임베딩은 다르다

라벨이 9개면 `100000000` ~ `000000001`. 이것도 여전히 **원핫**임

**임베딩은 그렇게 커지는 게 문제라서 나온 해결책**이라 방향이 정반대

| | 원핫 | 임베딩 |
|---|---|---|
| 차원 | **라벨 개수만큼** (1만 개면 1만 차원) | 내가 정함 (보통 50~300) |
| 값 | 0과 1뿐 (**희소**) | 실수 (**밀집**) |
| 값의 출처 | 규칙으로 만듦 | **학습으로 찾아냄** |
| 라벨 간 거리 | 전부 동일 | **의미에 따라 다름** |

→ 라벨 수천~수만 개(단어 등)일 때 임베딩을 씀. 분류 라벨 3~10개면 원핫으로 충분

## 3. 원핫 만드는 3가지 방법

```python
# 1) 텐서플로
from tensorflow.keras.utils import to_categorical
y = to_categorical(y)

# 2) 판다스
y = pd.get_dummies(y, dtype=float).values

# 3) sklearn
from sklearn.preprocessing import OneHotEncoder
ohe = OneHotEncoder(sparse_output=False)
y = ohe.fit_transform(y.reshape(-1, 1))
```

→ **원핫을 먼저 하고 split**하는 게 작업 횟수가 적음 (train/test 두 번 안 해도 됨)

### 각 방법의 주의점

| 방법 | 주의 |
|---|---|
| `to_categorical` | 라벨이 **0부터 시작해야** 함 (아래 참고) |
| `get_dummies` | 기본 dtype이 **`bool`** → `dtype=float` 필요. DataFrame이라 `.values` |
| `OneHotEncoder` | **2차원 입력** → `reshape(-1, 1)`. 기본이 희소행렬 → `sparse_output=False` 또는 `.toarray()` |

→ `reshape(-1, 1)`이 되는 조건: **내용이 안 바뀌고, 순서도 안 바뀜** (2일차 §3)

#### to_categorical의 함정 - 라벨이 1부터 시작할 때

`fetch_covtype`의 y는 **1~7**임 (0이 없음)

```python
y = np.array([1, 2, 3, 4, 5, 6, 7])

to_categorical(y).shape   # (7, 8)  <- 8열! 0번 열이 생김
to_categorical(y)[:, 0].sum()   # 0.0  <- 그 열은 전부 0, 영원히 안 쓰임
```

| 방법 | 라벨 1~7일 때 shape |
|---|---|
| `to_categorical` | **(7, 8)** ← 쓰지 않는 열이 하나 |
| `get_dummies` | (7, 7) |
| `OneHotEncoder` | (7, 7) |

→ 라벨이 0부터면 `to_categorical`도 문제없음. **1부터면 다른 방법을 쓰거나 `y - 1`을 해줘야 함**

In [ ]:
import numpy as np
import pandas as pd
from tensorflow.keras.utils import to_categorical
from sklearn.preprocessing import OneHotEncoder

y = np.array([1, 2, 3, 4, 5, 6, 7])          # covtype처럼 1부터 시작하는 라벨

print(to_categorical(y).shape)                # (7, 8)  <- 0번 열이 낭비됨
print(to_categorical(y)[:, 0].sum())          # 0.0     <- 전부 0

print(pd.get_dummies(y, dtype=float).values.shape)                        # (7, 7)
print(OneHotEncoder(sparse_output=False).fit_transform(y.reshape(-1,1)).shape)  # (7, 7)

# get_dummies의 기본 dtype은 bool - 케라스에 넣으려면 float로
print(pd.get_dummies(y).values.dtype)             # bool
print(pd.get_dummies(y, dtype=float).values.dtype)  # float64

### cf) LabelEncoder는 원핫이 아님

```python
le = LabelEncoder()
y = le.fit_transform(y)     # 'virginica' -> 2   (여전히 1차원 정수)
```

**대안이 아니라 전 단계**임

```
'virginica'  --LabelEncoder-->  2  --OneHotEncoder-->  [0, 0, 1]
   문자열                      정수                      원핫
```

| | 입력 | 출력 |
|---|---|---|
| `LabelEncoder` | **1차원** | 1차원 정수 |
| `OneHotEncoder` | **2차원** | 2차원 원핫 |

→ 기대하는 모양이 반대라 헷갈리기 쉬움. `LabelEncoder`에 `reshape(-1,1)`을 넣으면 `DataConversionWarning`

→ iris·wine·digits는 y가 이미 정수라 `LabelEncoder` 단계가 **불필요**

## 4. 출력층은 `softmax`

```python
model.add(Dense(3, activation='softmax'))    # 라벨 개수만큼
```

$$\text{softmax}(z_i) = \frac{e^{z_i}}{\sum_j e^{z_j}}$$

**출력의 합은 항상 정확히 1**

```
입력 [2.0, 1.0, 0.1]     -> [0.659, 0.242, 0.099]   합 1.0000000000
입력 [-5.0, -6.0, -7.0]  -> [0.665, 0.245, 0.090]   합 1.0000000000
입력 [100.0, 1.0, 1.0]   -> [1.000, 0.000, 0.000]   합 1.0000000000
```

→ 모든 항의 분모가 같고 분자를 다 더하면 그 분모가 되므로, 합이 1이 아닐 수 없음

### 합이 1 - 확률로 읽기 위해

출력을 **확률로 읽을 수 있게** 하려고

→ `[0.659, 0.242, 0.099]` = "setosa 65.9%, versicolor 24.2%, virginica 9.9%"

| | 각 값 | 합 |
|---|---|---|
| **sigmoid** (`Dense(1)`) | 0~1 | 값이 하나뿐 |
| **softmax** (`Dense(3)`) | 0~1 | **항상 1** |

→ sigmoid를 3개 노드에 각각 걸면 `[0.9, 0.8, 0.7]`처럼 합이 2.4가 될 수도 있음. **각 노드가 따로 계산하니까**

→ softmax는 분모를 공유해 **하나가 올라가면 나머지가 내려감**

## 5. argmax로 라벨 되돌리기

`predict` 결과는 확률 배열이라 `accuracy_score`에 바로 못 넣음

```python
y_predict = model.predict(x_test)          # (30, 3)  [[0.09, 0.24, 0.66], ...]

y_test    = np.argmax(y_test, axis=1)      # 원핫 -> 정수
y_predict = np.argmax(y_predict, axis=1)   # 확률 -> 정수

acc_score = accuracy_score(y_test, y_predict)
```

→ `argmax`는 **가장 큰 값의 위치(index)**를 돌려줌. 값(`0.66`)이 아니라 몇 번째(`2`)인지

→ `arg` = argument(위치). `axis=1`은 "행마다 가로로 훑어라"

### one-hot / softmax / argmax - 한 세트

```
정답 y:  2  ──one-hot──▶  [0, 0, 1]
                              ↕  categorical_crossentropy로 비교
예측:      [0.09, 0.24, 0.66]  ◀──softmax── Dense(3)
             └──argmax──▶  2
```

| 단계 | 하는 일 | 방향 |
|---|---|---|
| **one-hot** | 라벨 → 벡터 | 입력 전처리 |
| **softmax** | 점수 → 확률 | 모델 출력 |
| **argmax** | 확률 → 라벨 | 출력 후처리 |

→ 이진분류에서 `np.round`로 0.5를 자르던 것(7일차 §4)이 다중분류에서는 `argmax`가 됨

### cf) `evaluate`와 `predict`를 헷갈리면

```python
y_pred = model.evaluate(x_test)     # X
```
```
ValueError: None values not supported.
```

→ `evaluate`는 **채점 함수**라 정답 `y`가 있어야 함. 안 주면 `None`이 들어가 에러가 남

| | 인자 | 반환 |
|---|---|---|
| `evaluate(x, y)` | **x와 y 둘 다** | `[loss, acc]` |
| `predict(x)` | **x만** | 예측값 배열 |

## 6. 네 데이터셋 비교

| 파일 | 데이터 | shape | 라벨 | 결과 |
|---|---|---|---|---|
| `keras23_softmax1` | iris (붓꽃) | (150, 4) | 3 | acc 0.967 |
| `keras23_softmax2` | wine | (178, 13) | 3 | acc 0.972 · 120초 |
| `keras23_softmax3` | covtype (산림) | (581012, 54) | **7** | acc 0.900 · 339초 |
| `keras23_softmax4` | digits (손글씨) | (1797, 64) | **10** | acc 1.0 |

→ 코드 구조는 **네 개가 거의 같음.** 바뀌는 건 `input_dim`, 출력층 노드 수, `batch_size`뿐

### covtype - 규모와 불균형

```
라벨      1       2       3      4      5      6      7
개수  211840  283301  35754   2747   9493  17367  20510
```

→ 가장 많은 2번(283,301)과 가장 적은 4번(2,747)이 **103배** 차이

→ 7일차 §3의 클래스 불균형 상황. `stratify=y`로 비율은 유지했지만 **적은 클래스는 여전히 학습이 어려움**

→ 58만 행이라 `batch_size=2048`로 크게 잡음. 그래도 339초

### cf) `stratify`에 원핫을 넘겨도 되나

```python
y = pd.get_dummies(y, dtype=float).values   # (150, 3)
train_test_split(x, y, stratify=y)          # 2차원인데 동작함
```

→ 확인 결과 **정상 동작**. 행 전체를 하나의 조합으로 보고 층화함

→ 원핫은 행마다 1이 한 개뿐이라 결과적으로 라벨 기준 층화와 같아짐

### digits의 acc 1.0은 의심해볼 것

360개 test를 하나도 안 틀리는 건 흔치 않음

→ 4일차 §3의 "지나치게 높으면 의심할 것"에 해당

| 확인할 것 | |
|---|---|
| 과적합 | `val_loss` 곡선이 벌어지지 않았는지 |
| test가 너무 쉬움 | digits는 8×8 저해상도라 패턴이 단순한 편 |
| 계산 실수 | `argmax` 축이 맞는지, y_test를 되돌렸는지 |

→ `evaluate`의 `acc`와 `accuracy_score`가 **같은 값인지** 대조해보면 계산 실수는 걸러짐